# Person 3 — Fine-tuning the shared Week 7 SLM

Training starts from Person 1's original checkpoint
`a7/slm/checkpoints/selected_baseline.pt`. It does not start from a new
initialization and it does not load a RAG-updated model.

The ten examples and five evaluation questions come from Person 2's
knowledge base: laundry day, trash day, guest towels, spare bulbs, and
dish soap. Each baseline prompt is the question plus one newline, the same
string Person 2 used with no retrieved document.

Generation is CPU, two threads, seed 42, temperature 0.7, and 100 new
characters. Re-run with `python -m a7.person3.finetune_slm`.


In [ ]:
from a7.person3.finetune_slm import run
import json
payload = run()
print(json.dumps({
    "architecture": payload["architecture"],
    "example_count": payload["example_count"],
    "learning_rate": payload["fine_tune"]["learning_rate"],
    "steps": payload["fine_tune"]["steps"],
    "initial_loss": payload["initial_loss"],
    "final_loss": payload["final_loss"],
    "training_seconds": payload["training_seconds"],
    "weights_updated": payload["weights_updated"],
    "original_state_sha256": payload["original_state_sha256"],
    "finetuned_state_sha256": payload["finetuned_state_sha256"],
}, indent=2))


## Before and after

Scores count how many expected fact phrases appear in the continuation.
A hit does not mean the sentence is a clean answer. The model still mixes
facts from other documents because it only sees the last 32 characters.


In [ ]:
import json
from pathlib import Path
payload = json.loads(Path("a7/person3/results/finetune_results.json").read_text())
print(f"{'Q':<4} {'Before':<70} {'After'}")
for before, after in zip(payload["before"], payload["after"]):
    print(before["question_id"], before["factual_accuracy"], after["factual_accuracy"])
    print("  prompt:", before["prompt"])
    print("  before:", before["response"].replace("\n", " "))
    print("  after: ", after["response"].replace("\n", " "))


## Part 3 analysis

The five pre-fine-tune continuations match Person 2's exported baselines
exactly, and the starting state hash is the shared checkpoint hash.
Training loss fell from 4.69 to 0.22 in 300 AdamW steps. The state hash
changed, so the weights were updated.

Responses changed. Guest-towel and spare-bulb continuations now contain
`in the hall` and `in the shed`. Laundry day, trash day, and dish soap
still do not state `monday`, `friday`, or `under the sink`. The model
copies phrases from other documents, including `under the shed` for the
soap question.

Fine-tuning improved some factual phrases and left others wrong. A
fine-tuned SLM can still generate incorrect or unsupported text. The
32-character window cannot hold a full question and its answer at once,
so later facts leak into earlier answers.
